# Sprite Generator - Transformer Training
Trains conditional transformer prior on VQ-VAE tokens. Loads VQ-VAE from HF Hub.

In [ ]:
import os, sys, torch, json
from pathlib import Path

!pip install -q datasets huggingface_hub torchvision pillow tqdm

!git clone https://github.com/MANI8148/sprite-generator.git /kaggle/working/sprite-gen
sys.path.insert(0, '/kaggle/working/sprite-gen')

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_DATASET = os.environ.get("HF_DATASET", "darklord8777/sprites")
HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "darklord8777/sprite-generator-model")
EPOCHS = int(os.environ.get("EPOCHS", "100"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "64"))
LR = float(os.environ.get("LR", "3e-4"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load VQ-VAE from HF Hub

In [ ]:
from huggingface_hub import hf_hub_download
from models.vqvae.model import VQVAE

ckpt_path = hf_hub_download(HF_MODEL_REPO, "vqvae_latest.pt", token=HF_TOKEN)
ckpt = torch.load(ckpt_path, map_location=device)
# Infer num_embeddings from checkpoint config or embedding weight shape
num_emb = ckpt.get("config", {}).get("num_embeddings")
if num_emb is None:
    num_emb = ckpt["model_state"]["quantizer.embedding.weight"].size(0)
vqvae = VQVAE(num_embeddings=num_emb).to(device)
vqvae.load_state_dict(ckpt["model_state"])
vqvae.eval()
print(f"VQ-VAE loaded from HF Hub (codebook size: {num_emb})")

## Build token dataset
Encode all sprites through VQ-VAE once, cache token sequences.

In [ ]:
import numpy as np
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm

dataset = load_dataset(HF_DATASET, split="train")
print(f"Dataset size: {len(dataset)}")

CLASS_VOCAB = ["unknown","character","item","tile","enemy","player","weapon","food",
               "vehicle","building","decoration","effect","projectile","animal","plant",
               "furniture","tool","accessory","ui_element","terrain"]
ACTION_VOCAB = ["idle","walk","run","attack","jump","hurt","death","block","shoot",
                "cast","interact","fly","swim","climb"]
DIRECTION_VOCAB = ["front","back","left","right","front_left","front_right","back_left","back_right"]

def encode_condition(value, vocab):
    try: return vocab.index(value)
    except ValueError: return 0

all_tokens, all_class, all_action, all_direction = [], [], [], []

for item in tqdm(dataset, desc="Encoding tokens"):
    img = item["image"].convert("RGBA").resize((32, 32), Image.NEAREST)
    img_tensor = torch.tensor(np.array(img).astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        tokens = vqvae.encode_to_indices(img_tensor).squeeze(0).cpu()
    all_tokens.append(tokens)
    all_class.append(encode_condition(item.get("class","unknown"), CLASS_VOCAB))
    all_action.append(encode_condition(item.get("action","idle"), ACTION_VOCAB))
    all_direction.append(encode_condition(item.get("direction","front"), DIRECTION_VOCAB))

token_data = {
    "tokens": torch.stack(all_tokens).numpy(),
    "class": np.array(all_class),
    "action": np.array(all_action),
    "direction": np.array(all_direction),
}
np.savez("/kaggle/working/token_dataset.npz", **token_data)
print(f"Saved {len(all_tokens)} token sequences")

## Train Transformer

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from models.transformer.model import SpriteTransformer

# Load cached token data
data = np.load("/kaggle/working/token_dataset.npz")
tokens = torch.tensor(data["tokens"])
class_ids = torch.tensor(data["class"])
action_ids = torch.tensor(data["action"])
direction_ids = torch.tensor(data["direction"])

train_dataset = TensorDataset(tokens, class_ids, action_ids, direction_ids)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

model = SpriteTransformer(
    vocab_size=num_emb,
    condition_vocab_size=64,
    d_model=512,
    n_layers=12,
    n_heads=8,
    max_seq_len=tokens.shape[1] + 1,
    dropout=0.1,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
warmup_epochs = 5
import math
def warmup_cosine(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    return 0.5 * (1 + math.cos((epoch - warmup_epochs) / (EPOCHS - warmup_epochs) * math.pi))
scheduler = optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine)

checkpoint_dir = Path("/kaggle/working/checkpoints/transformer")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_tokens, c, a, d in pbar:
        batch_tokens, c, a, d = batch_tokens.to(device), c.to(device), a.to(device), d.to(device)
        optimizer.zero_grad()
        logits = model(batch_tokens, c, a, d)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_tokens.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
    
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}: loss={avg_loss:.6f}")
    
    # Save checkpoint every 10 epochs and push to HF
    if (epoch + 1) % 10 == 0:
        ckpt_path = checkpoint_dir / f"transformer_epoch_{epoch+1:03d}.pt"
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss": avg_loss,
        }, ckpt_path)
        print(f"Saved {ckpt_path}")
        if HF_TOKEN:
            from huggingface_hub import HfApi
            HfApi(token=HF_TOKEN).upload_file(
                path_or_fileobj=str(ckpt_path),
                path_in_repo=f"transformer_epoch_{epoch+1:03d}.pt",
                repo_id=HF_MODEL_REPO, repo_type="model",
            )
            HfApi(token=HF_TOKEN).upload_file(
                path_or_fileobj=str(ckpt_path),
                path_in_repo="transformer_latest.pt",
                repo_id=HF_MODEL_REPO, repo_type="model",
            )
            print(f"  -> Pushed to HF (epoch {epoch+1})")

# Save and push final
ckpt_final = checkpoint_dir / "transformer_final.pt"
torch.save({
    "epoch": EPOCHS - 1,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "loss": avg_loss,
}, ckpt_final)
print(f"Saved {ckpt_final}")
if HF_TOKEN:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(
        path_or_fileobj=str(ckpt_final),
        path_in_repo="transformer_latest.pt",
        repo_id=HF_MODEL_REPO, repo_type="model",
    )
    print("Final checkpoint pushed to HF")
print("Training complete!")

## Push checkpoint and training_complete marker to HF Hub

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    
    # Push transformer checkpoint
    api.upload_file(
        path_or_fileobj=str(checkpoint_dir / "transformer_final.pt"),
        path_in_repo="transformer_latest.pt",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )
    print("Transformer checkpoint pushed to HF Hub")
    
    # Push training_complete marker
    api.upload_file(
        path_or_fileobj=json.dumps({
            "status": "complete",
            "model": "transformer",
            "epochs": EPOCHS,
        }).encode(),
        path_in_repo="transformer_complete.json",
        repo_id=HF_MODEL_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )
    print("Training marker pushed")
else:
    print("Skipping HF push: no HF_TOKEN")